# Practical Fine-Tuning with HuggingFace

**Module 09: Fine-Tuning**  
**Objective**: Apply PEFT methods using production tools

Production fine-tuning:
- HuggingFace PEFT library
- Trainer API
- Dataset preparation
- Evaluation and metrics
- Distributed training

## What You'll Learn
1. Using PEFT library (LoRA, QLoRA)
2. HuggingFace Trainer API
3. Dataset preprocessing
4. Training strategies
5. Evaluation metrics
6. Deployment of fine-tuned models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)

## 1. PEFT Library Overview

In [ ]:
# HuggingFace PEFT library examples

peft_examples = """
# ================================================
# Installation
# ================================================
# pip install peft transformers accelerate bitsandbytes


# ================================================
# Basic LoRA Setup
# ================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# Load base model
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Configure LoRA
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=16,  # Scaling factor
    target_modules=["c_attn"],  # Apply to attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Output: trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.24


# ================================================
# QLoRA: 4-bit Quantization + LoRA
# ================================================
from transformers import BitsAndBytesConfig

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True  # Nested quantization
)

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    quantization_config=bnb_config,
    device_map="auto"
)

# Add LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)


# ================================================
# Training with Trainer API
# ================================================
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16
    learning_rate=2e-4,
    fp16=True,  # Mixed precision
    logging_steps=10,
    save_steps=100,
    evaluation_strategy="steps",
    eval_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

# Train
trainer.train()


# ================================================
# Save and Load LoRA Adapters
# ================================================
# Save only LoRA weights (tiny file, ~few MB)
model.save_pretrained("./lora_adapters")

# Load later
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("gpt2")
model = PeftModel.from_pretrained(base_model, "./lora_adapters")

# Merge and save full model
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged_model")


# ================================================
# Multiple Adapters (Task Switching)
# ================================================
# Train adapter for Task A
model.save_pretrained("./adapter_task_a")

# Train adapter for Task B
model.save_pretrained("./adapter_task_b")

# Switch between tasks
base_model = AutoModelForCausalLM.from_pretrained("gpt2")
model = PeftModel.from_pretrained(base_model, "./adapter_task_a")

# Switch to Task B
model.load_adapter("./adapter_task_b", adapter_name="task_b")
model.set_adapter("task_b")


# ================================================
# Prefix Tuning
# ================================================
from peft import PrefixTuningConfig

prefix_config = PrefixTuningConfig(
    task_type="CAUSAL_LM",
    num_virtual_tokens=20,
    prefix_projection=True
)

model = get_peft_model(model, prefix_config)


# ================================================
# IA3 (Infused Adapter by Inhibiting and Amplifying Inner Activations)
# ================================================
from peft import IA3Config

ia3_config = IA3Config(
    task_type="CAUSAL_LM",
    target_modules=["k_proj", "v_proj", "down_proj"],
    feedforward_modules=["down_proj"]
)

model = get_peft_model(model, ia3_config)
"""

print("HuggingFace PEFT Library Guide:\n")
print(peft_examples)

## 2. Dataset Preparation

In [ ]:
# Dataset preparation examples

dataset_examples = """
# ================================================
# Simple Text Classification
# ================================================
from datasets import load_dataset

# Load dataset
dataset = load_dataset("imdb")

# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)


# ================================================
# Instruction Fine-tuning Format
# ================================================
# Alpaca/Instruct format
instruction_template = \"\"\"Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{response}\"\"\"

def format_instruction(example):
    text = instruction_template.format(
        instruction=example["instruction"],
        response=example["output"]
    )
    return tokenizer(text, truncation=True, max_length=1024)

dataset = dataset.map(format_instruction)


# ================================================
# Chat Format (Multi-turn)
# ================================================
# Format: <|user|> ... <|assistant|> ...
def format_chat(example):
    conversation = []
    for turn in example["messages"]:
        role = turn["role"]
        content = turn["content"]
        conversation.append(f"<|{role}|> {content}")
    
    text = "\\n".join(conversation) + "<|assistant|>"
    return tokenizer(text, truncation=True, max_length=2048)


# ================================================
# Custom Dataset from JSON
# ================================================
import json

# Load custom data
with open("training_data.json") as f:
    data = json.load(f)

# Convert to HuggingFace Dataset
from datasets import Dataset

dataset = Dataset.from_dict({
    "text": [item["text"] for item in data],
    "label": [item["label"] for item in data]
})


# ================================================
# Streaming for Large Datasets
# ================================================
# Don't load entire dataset into memory
dataset = load_dataset("oscar", "unshuffled_deduplicated_en", streaming=True)

# Process in batches
for batch in dataset.take(1000):
    # Process batch
    pass


# ================================================
# Data Collator (Dynamic Padding)
# ================================================
from transformers import DataCollatorForLanguageModeling

# For causal LM (GPT-style)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # mlm=True for masked LM (BERT-style)
)

# For sequence classification
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
"""

print("Dataset Preparation Patterns:\n")
print(dataset_examples)

## 3. Training Strategies

In [ ]:
def visualize_training_strategies():
    """Compare different training approaches."""
    
    # Simulated training curves
    epochs = np.arange(0, 10, 0.1)
    
    # Standard training
    standard_loss = 2.5 * np.exp(-0.3 * epochs) + 0.5
    
    # With gradient accumulation (stabler)
    grad_accum_loss = 2.5 * np.exp(-0.35 * epochs) + 0.45
    
    # With warmup (better start)
    warmup_loss = 3.0 * np.exp(-0.4 * epochs) + 0.4
    
    # With learning rate scheduling
    scheduled_loss = 2.5 * np.exp(-0.38 * epochs) + 0.42
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Training curves
    axes[0, 0].plot(epochs, standard_loss, label='Standard', linewidth=2)
    axes[0, 0].plot(epochs, grad_accum_loss, label='+ Gradient Accumulation', linewidth=2)
    axes[0, 0].plot(epochs, warmup_loss, label='+ Warmup', linewidth=2)
    axes[0, 0].plot(epochs, scheduled_loss, label='+ LR Scheduling', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Curves with Different Strategies')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # Learning rate schedules
    steps = np.arange(0, 1000)
    
    # Constant
    lr_constant = np.ones_like(steps) * 1e-4
    
    # Linear warmup + decay
    warmup_steps = 100
    lr_warmup = np.concatenate([
        np.linspace(0, 1e-4, warmup_steps),
        np.linspace(1e-4, 1e-5, len(steps) - warmup_steps)
    ])
    
    # Cosine annealing
    lr_cosine = 1e-5 + (1e-4 - 1e-5) * (1 + np.cos(np.pi * steps / len(steps))) / 2
    
    # Exponential decay
    lr_exp = 1e-4 * np.exp(-steps / 300)
    
    axes[0, 1].plot(steps, lr_constant, label='Constant', linewidth=2)
    axes[0, 1].plot(steps, lr_warmup, label='Warmup + Linear', linewidth=2)
    axes[0, 1].plot(steps, lr_cosine, label='Cosine Annealing', linewidth=2)
    axes[0, 1].plot(steps, lr_exp, label='Exponential Decay', linewidth=2)
    axes[0, 1].set_xlabel('Training Steps')
    axes[0, 1].set_ylabel('Learning Rate')
    axes[0, 1].set_title('Learning Rate Schedules')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    axes[0, 1].set_yscale('log')
    
    # Batch size vs memory
    batch_sizes = [1, 2, 4, 8, 16, 32]
    memory_usage = [size * 2 for size in batch_sizes]  # Simplified
    training_speed = [1/size * 100 for size in batch_sizes]  # Steps per second
    
    ax1 = axes[1, 0]
    ax2 = ax1.twinx()
    
    line1 = ax1.plot(batch_sizes, memory_usage, marker='o', color='red', linewidth=2, label='Memory')
    line2 = ax2.plot(batch_sizes, training_speed, marker='s', color='blue', linewidth=2, label='Speed')
    
    ax1.set_xlabel('Batch Size')
    ax1.set_ylabel('GPU Memory (GB)', color='red')
    ax2.set_ylabel('Training Speed (steps/sec)', color='blue')
    ax1.set_title('Batch Size Trade-offs')
    ax1.grid(alpha=0.3)
    
    # Combined legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    
    # Gradient accumulation illustration
    steps_per_update = [1, 2, 4, 8]
    effective_batch = [bs * 4 for bs in steps_per_update]  # Assume base batch=4
    memory_per_step = [4 + 1 for _ in steps_per_update]  # Constant memory
    
    axes[1, 1].bar(range(len(steps_per_update)), effective_batch, 
                   color='lightblue', alpha=0.7, label='Effective Batch Size')
    axes[1, 1].set_xticks(range(len(steps_per_update)))
    axes[1, 1].set_xticklabels([f'{s} steps' for s in steps_per_update])
    axes[1, 1].set_xlabel('Gradient Accumulation Steps')
    axes[1, 1].set_ylabel('Effective Batch Size')
    axes[1, 1].set_title('Gradient Accumulation: Larger Batch, Same Memory')
    axes[1, 1].legend()
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    # Annotate memory stays constant
    for i, (steps, batch) in enumerate(zip(steps_per_update, effective_batch)):
        axes[1, 1].text(i, batch + 1, f'{batch}', ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print("\nTraining Strategy Recommendations:\n")
    print("1. Learning Rate:")
    print("   • LoRA: 1e-4 to 5e-4 (higher than full fine-tuning)")
    print("   • Full fine-tuning: 1e-5 to 5e-5")
    print("   • Use warmup (5-10% of total steps)")
    
    print("\n2. Batch Size:")
    print("   • Larger = stabler gradients but more memory")
    print("   • Use gradient accumulation if memory-limited")
    print("   • Effective batch size: 16-128 for most tasks")
    
    print("\n3. Epochs:")
    print("   • 3-5 epochs typical for instruction tuning")
    print("   • Watch for overfitting (eval loss increases)")
    print("   • Use early stopping")
    
    print("\n4. Gradient Clipping:")
    print("   • Clip norm to 1.0 to prevent exploding gradients")
    print("   • Especially important for LoRA")

visualize_training_strategies()

## 4. Evaluation Metrics

In [ ]:
def implement_evaluation_metrics():
    """Implement common evaluation metrics."""
    
    # Example predictions and references
    predictions = [
        "The movie was amazing and I loved it.",
        "Paris is the capital of France.",
        "def add(a, b):\\n    return a + b"
    ]
    
    references = [
        "The movie was great and I enjoyed it.",
        "Paris is France's capital city.",
        "def add(a, b):\\n    return a + b"
    ]
    
    print("Evaluation Metrics for Fine-tuned Models:\n")
    print("="*70)
    
    # 1. Exact Match
    print("\n1. EXACT MATCH")
    print("   Use case: Classification, closed-form QA")
    exact_matches = sum(p == r for p, r in zip(predictions, references))
    print(f"   Score: {exact_matches}/{len(predictions)} = {exact_matches/len(predictions)*100:.1f}%")
    
    # 2. Token-level accuracy
    print("\n2. TOKEN ACCURACY")
    print("   Use case: Sequence generation")
    
    def token_accuracy(pred, ref):
        pred_tokens = pred.split()
        ref_tokens = ref.split()
        max_len = max(len(pred_tokens), len(ref_tokens))
        matches = sum(p == r for p, r in zip(pred_tokens, ref_tokens))
        return matches / max_len
    
    token_accs = [token_accuracy(p, r) for p, r in zip(predictions, references)]
    print(f"   Average: {np.mean(token_accs)*100:.1f}%")
    
    # 3. BLEU score (n-gram precision)
    print("\n3. BLEU SCORE")
    print("   Use case: Translation, generation")
    print("   Formula: Geometric mean of n-gram precisions")
    
    # Simplified BLEU-1 (unigram)
    def bleu_1(pred, ref):
        pred_tokens = set(pred.split())
        ref_tokens = set(ref.split())
        if len(pred_tokens) == 0:
            return 0
        matches = len(pred_tokens & ref_tokens)
        return matches / len(pred_tokens)
    
    bleu_scores = [bleu_1(p, r) for p, r in zip(predictions, references)]
    print(f"   BLEU-1 Average: {np.mean(bleu_scores)*100:.1f}%")
    
    # 4. ROUGE (recall-oriented)
    print("\n4. ROUGE SCORE")
    print("   Use case: Summarization")
    print("   Formula: Recall of n-grams")
    
    def rouge_1(pred, ref):
        pred_tokens = set(pred.split())
        ref_tokens = set(ref.split())
        if len(ref_tokens) == 0:
            return 0
        matches = len(pred_tokens & ref_tokens)
        return matches / len(ref_tokens)
    
    rouge_scores = [rouge_1(p, r) for p, r in zip(predictions, references)]
    print(f"   ROUGE-1 Average: {np.mean(rouge_scores)*100:.1f}%")
    
    # 5. Perplexity
    print("\n5. PERPLEXITY")
    print("   Use case: Language modeling quality")
    print("   Formula: exp(average cross-entropy loss)")
    
    # Simulated cross-entropy losses
    losses = [0.5, 0.8, 0.3]
    perplexities = [np.exp(loss) for loss in losses]
    print(f"   Average perplexity: {np.mean(perplexities):.2f}")
    print(f"   (Lower is better, random = vocab_size)")
    
    print("\n" + "="*70)
    
    # Visualize metrics
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Metric comparison for examples
    metrics_data = {
        'Example 1': [bleu_scores[0]*100, rouge_scores[0]*100, token_accs[0]*100],
        'Example 2': [bleu_scores[1]*100, rouge_scores[1]*100, token_accs[1]*100],
        'Example 3': [bleu_scores[2]*100, rouge_scores[2]*100, token_accs[2]*100]
    }
    
    x = np.arange(3)
    width = 0.25
    
    for i, (label, values) in enumerate(metrics_data.items()):
        axes[0].bar(x + i*width, values, width, label=label)
    
    axes[0].set_ylabel('Score (%)')
    axes[0].set_title('Metrics Comparison Across Examples')
    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels(['BLEU-1', 'ROUGE-1', 'Token Acc'])
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Perplexity over training
    training_steps = np.linspace(0, 1000, 100)
    train_perplexity = 150 * np.exp(-training_steps / 200) + 5
    val_perplexity = 150 * np.exp(-training_steps / 250) + 8
    
    axes[1].plot(training_steps, train_perplexity, label='Train', linewidth=2)
    axes[1].plot(training_steps, val_perplexity, label='Validation', linewidth=2)
    axes[1].axhline(y=10, color='red', linestyle='--', label='Target', alpha=0.5)
    axes[1].set_xlabel('Training Steps')
    axes[1].set_ylabel('Perplexity')
    axes[1].set_title('Perplexity During Training')
    axes[1].legend()
    axes[1].set_yscale('log')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

implement_evaluation_metrics()

## 5. Complete Training Example

In [ ]:
# Complete fine-tuning workflow

complete_workflow = """
# ========================================
# COMPLETE FINE-TUNING WORKFLOW
# ========================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# ========================================
# 1. Setup
# ========================================

model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load model in 4-bit
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# ========================================
# 2. Configure LoRA
# ========================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ========================================
# 3. Prepare Dataset
# ========================================

# Load dataset (e.g., Alpaca format)
dataset = load_dataset("tatsu-lab/alpaca")

# Format prompts
def format_prompts(example):
    instruction = example["instruction"]
    input_text = example["input"]
    output = example["output"]
    
    if input_text:
        prompt = f\"\"\"Below is an instruction with context. Write a response.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output}\"\"\"
    else:
        prompt = f\"\"\"Below is an instruction. Write a response.

### Instruction:
{instruction}

### Response:
{output}\"\"\"
    
    return {"text": prompt}

# Apply formatting
formatted_dataset = dataset.map(format_prompts)

# Tokenize
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = formatted_dataset.map(
    tokenize,
    remove_columns=formatted_dataset["train"].column_names
)

# ========================================
# 4. Training Arguments
# ========================================

training_args = TrainingArguments(
    output_dir="./llama2-7b-alpaca-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",  # Optimized for QLoRA
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    evaluation_strategy="steps",
    eval_steps=100,
    fp16=True,
    max_grad_norm=0.3,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="tensorboard"
)

# ========================================
# 5. Create Trainer
# ========================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# ========================================
# 6. Train
# ========================================

print("Starting training...")
trainer.train()

print("Training complete!")

# ========================================
# 7. Save Model
# ========================================

# Save LoRA adapters
model.save_pretrained("./lora_adapters")
tokenizer.save_pretrained("./lora_adapters")

# Optionally merge and save full model
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged_model")

# ========================================
# 8. Inference
# ========================================

# Load adapted model
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "./lora_adapters")

# Generate
prompt = \"\"\"Below is an instruction. Write a response.

### Instruction:
Write a Python function to calculate fibonacci numbers.

### Response:
\"\"\"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.95,
    do_sample=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)
"""

print("Complete Fine-tuning Workflow:\n")
print(complete_workflow)

## 6. Troubleshooting Common Issues

In [ ]:
troubleshooting_guide = """
# ========================================
# COMMON ISSUES AND SOLUTIONS
# ========================================

# 1. OUT OF MEMORY (OOM)
# ========================================
Problem: CUDA out of memory error

Solutions:
  a) Reduce batch size
     per_device_train_batch_size=2  # instead of 4
  
  b) Use gradient accumulation
     gradient_accumulation_steps=8  # Effectively larger batch
  
  c) Enable gradient checkpointing
     gradient_checkpointing=True  # Trade compute for memory
  
  d) Use 8-bit or 4-bit quantization
     load_in_8bit=True
  
  e) Reduce sequence length
     max_length=512  # instead of 1024


# 2. TRAINING TOO SLOW
# ========================================
Problem: Training takes forever

Solutions:
  a) Use larger batch size (if memory allows)
     per_device_train_batch_size=8
  
  b) Enable mixed precision
     fp16=True  # or bf16=True
  
  c) Use faster optimizer
     optim="adamw_torch_fused"
  
  d) Reduce logging frequency
     logging_steps=100  # instead of 10
  
  e) Use flash attention (if available)
     attn_implementation="flash_attention_2"


# 3. MODEL NOT LEARNING (Loss not decreasing)
# ========================================
Problem: Loss stays flat or high

Solutions:
  a) Increase learning rate
     learning_rate=5e-4  # Higher for LoRA
  
  b) Remove warmup (or reduce)
     warmup_ratio=0.0
  
  c) Check data preprocessing
     # Ensure labels are set correctly
     result["labels"] = result["input_ids"].copy()
  
  d) Increase LoRA rank
     r=16  # instead of 8
  
  e) Add more target modules
     target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]


# 4. OVERFITTING (Train loss << Eval loss)
# ========================================
Problem: Model memorizes training data

Solutions:
  a) Reduce epochs
     num_train_epochs=1
  
  b) Add dropout
     lora_dropout=0.1
  
  c) Use weight decay
     weight_decay=0.01
  
  d) Early stopping
     load_best_model_at_end=True
     metric_for_best_model="eval_loss"
  
  e) Get more training data
     # Augment or find more examples


# 5. GENERATION QUALITY POOR
# ========================================
Problem: Model outputs nonsense or repetitive text

Solutions:
  a) Adjust generation parameters
     temperature=0.7      # Lower = more focused
     top_p=0.9           # Nucleus sampling
     repetition_penalty=1.2
  
  b) Longer training
     num_train_epochs=5
  
  c) Better prompting
     # Use clear instruction format
     # Add examples (few-shot)
  
  d) Check tokenizer padding
     tokenizer.pad_token = tokenizer.eos_token
     tokenizer.padding_side = "right"
  
  e) Fine-tune on higher quality data
     # Filter training examples


# 6. LORA ADAPTERS NOT LOADING
# ========================================
Problem: Can't load saved adapters

Solutions:
  a) Check saved files
     ls ./lora_adapters
     # Should have: adapter_config.json, adapter_model.bin
  
  b) Load with correct base model
     base_model = AutoModelForCausalLM.from_pretrained("gpt2")
     model = PeftModel.from_pretrained(base_model, "./lora_adapters")
  
  c) Check PEFT version
     pip install --upgrade peft
  
  d) Verify model architecture matches
     # Base model must match original


# 7. DISTRIBUTED TRAINING HANGS
# ========================================
Problem: Multi-GPU training freezes

Solutions:
  a) Set environment variables
     export NCCL_P2P_DISABLE=1
     export NCCL_IB_DISABLE=1
  
  b) Use accelerate launch
     accelerate launch train.py
  
  c) Check GPU communication
     nvidia-smi topo -m
  
  d) Reduce number of GPUs
     CUDA_VISIBLE_DEVICES=0,1  # Use only 2 GPUs
"""

print("Troubleshooting Guide:\n")
print(troubleshooting_guide)

## 7. Deployment Considerations

In [ ]:
def compare_deployment_options():
    """Compare different deployment strategies."""
    
    options = {
        'LoRA Adapters': {
            'size_mb': 50,
            'load_time': 2,
            'memory_gb': 14,
            'flexibility': 100,
            'description': 'Swap adapters dynamically'
        },
        'Merged Model': {
            'size_mb': 14000,
            'load_time': 10,
            'memory_gb': 14,
            'flexibility': 30,
            'description': 'Single model file'
        },
        'Quantized (8-bit)': {
            'size_mb': 7000,
            'load_time': 8,
            'memory_gb': 7,
            'flexibility': 50,
            'description': 'Half memory, slight quality loss'
        },
        'Quantized (4-bit)': {
            'size_mb': 3500,
            'load_time': 5,
            'memory_gb': 3.5,
            'flexibility': 50,
            'description': 'Quarter memory, ~3% quality loss'
        }
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    names = list(options.keys())
    colors = ['skyblue', 'lightcoral', 'lightgreen', 'gold']
    
    # Size comparison
    sizes = [options[name]['size_mb'] for name in names]
    axes[0, 0].barh(names, sizes, color=colors)
    axes[0, 0].set_xlabel('Model Size (MB)')
    axes[0, 0].set_title('Storage Requirements')
    axes[0, 0].set_xscale('log')
    axes[0, 0].grid(axis='x', alpha=0.3)
    
    # Memory usage
    memory = [options[name]['memory_gb'] for name in names]
    axes[0, 1].barh(names, memory, color=colors)
    axes[0, 1].set_xlabel('GPU Memory (GB)')
    axes[0, 1].set_title('Runtime Memory')
    axes[0, 1].grid(axis='x', alpha=0.3)
    
    # Load time
    load_times = [options[name]['load_time'] for name in names]
    axes[1, 0].barh(names, load_times, color=colors)
    axes[1, 0].set_xlabel('Load Time (seconds)')
    axes[1, 0].set_title('Model Loading Speed')
    axes[1, 0].grid(axis='x', alpha=0.3)
    
    # Flexibility score
    flexibility = [options[name]['flexibility'] for name in names]
    axes[1, 1].barh(names, flexibility, color=colors)
    axes[1, 1].set_xlabel('Flexibility Score')
    axes[1, 1].set_title('Deployment Flexibility')
    axes[1, 1].set_xlim(0, 110)
    axes[1, 1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print("Deployment Recommendations:")
    print("="*70)
    
    for name, props in options.items():
        print(f"\n{name.upper()}")
        print(f"  Size: {props['size_mb']}MB")
        print(f"  Memory: {props['memory_gb']}GB")
        print(f"  Load time: {props['load_time']}s")
        print(f"  Use case: {props['description']}")
    
    print("\n" + "="*70)
    print("\nBest Practices:")
    print("  1. Development: Use LoRA adapters for experimentation")
    print("  2. Production (single task): Merge and quantize to 8-bit")
    print("  3. Production (multi-task): Keep adapters, load dynamically")
    print("  4. Edge deployment: 4-bit quantization")
    print("="*70)

compare_deployment_options()

## Summary

You've mastered practical fine-tuning:
- ✅ HuggingFace PEFT library (LoRA, QLoRA, Prefix, IA3)
- ✅ Trainer API configuration
- ✅ Dataset preparation and formatting
- ✅ Training strategies and hyperparameters
- ✅ Evaluation metrics
- ✅ Troubleshooting common issues
- ✅ Deployment considerations

**Key Takeaways**:
1. **QLoRA** enables fine-tuning 70B models on single consumer GPU
2. **Learning rate** for LoRA: 2e-4 (2-10× higher than full fine-tuning)
3. **Gradient accumulation** allows large effective batch size with limited memory
4. **Evaluation**: Use task-specific metrics + perplexity
5. **Deployment**: LoRA adapters (~50MB) vs full model (~14GB)

**Production Checklist**:
- [ ] Use cosine LR schedule with warmup
- [ ] Enable gradient checkpointing if memory-limited
- [ ] Monitor both train and validation metrics
- [ ] Save multiple checkpoints (not just final)
- [ ] Test generation quality manually
- [ ] Version control adapter configs
- [ ] Benchmark inference speed

**Next Module**: RAG (Retrieval-Augmented Generation)

## Exercises

1. **Fine-tune on custom dataset**: Prepare and fine-tune on your own data
2. **Hyperparameter search**: Compare r=[4,8,16,32] and lr=[1e-4,2e-4,5e-4]
3. **Multi-adapter system**: Train adapters for 3 different tasks, switch between them